## Sieć neuronowa do rozpoznawania instrumentów

### Biblioteki

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

### Pobranie danych wejściowych z mel-spektogramu

In [2]:
spektogramy = Path("spektrogramy")
csv = spektogramy / "metadata.csv"

data = pd.read_csv(csv)

In [3]:
etykiety = pd.read_csv("slakh_trial/instrumenty.csv")

print(etykiety.shape)
print(etykiety.columns.tolist())

(1710, 8)
['Track', 'Acoustic Guitar (steel)', 'Alto Sax', 'Bright Acoustic Piano', 'Drums', 'Electric Bass (finger)', 'Flute', 'String Ensemble 1']


In [4]:
instrumenty = [
    "Acoustic Guitar (steel)",
    "Alto Sax",
    "Bright Acoustic Piano",
    "Drums",
    "Electric Bass (finger)",
    "Flute",
    "String Ensemble 1"
]

data = data.merge(
    etykiety[["Track"] + instrumenty],
    left_on="stem_name",
    right_on="Track",
    how="left",
    validate="many_to_one"
)

print("Liczba rekordów:", len(data))
print("Brakujące etykiety:", data[instrumenty].isna().any(axis=1).sum())
print(data[["stem_name"] + instrumenty].head())

Liczba rekordów: 20
Brakujące etykiety: 0
    stem_name  Acoustic Guitar (steel)  Alto Sax  Bright Acoustic Piano  \
0  Track00011                        1         1                      1   
1  Track00012                        0         0                      0   
2  Track00013                        1         0                      0   
3  Track00014                        1         0                      0   
4  Track00015                        0         0                      1   

   Drums  Electric Bass (finger)  Flute  String Ensemble 1  
0      1                       1      0                  0  
1      1                       0      0                  0  
2      1                       0      1                  0  
3      1                       1      0                  0  
4      1                       0      0                  0  


In [5]:
plik = spektogramy / data.iloc[0]["file_path"].replace("\\", "/")
spec = np.load(plik)

print("Plik:", plik)
print("Kształt:", spec.shape)
print("Minimum:", spec.min())
print("Maksimum:", spec.max())
print("Liczba niezerowych wartości:", np.count_nonzero(spec))

Plik: spektrogramy/test/Track00011.npy
Kształt: (80, 346)
Minimum: -80.0
Maksimum: 0.0
Liczba niezerowych wartości: 27679


In [6]:
for zbior in ["train", "val", "test"]:
    ksztalty = set()

    for sciezka in data.loc[data["set"] == zbior, "file_path"]:
        sciezka = sciezka.replace("\\", "/")
        spec = np.load(spektogramy / sciezka)
        ksztalty.add(spec.shape)

    print(zbior, "— wymiary spektrogramów:", ksztalty)

train — wymiary spektrogramów: {(80, 346)}
val — wymiary spektrogramów: {(80, 346)}
test — wymiary spektrogramów: {(80, 346)}


In [7]:

class InstrumentDataset(Dataset):
    def __init__(self, data, root, zbior):
        self.data = data[data["set"] == zbior].reset_index(drop=True)
        self.root = root

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        sciezka = row["file_path"].replace("\\", "/")
        spec = np.load(self.root / sciezka)

        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)

        label = row[instrumenty].to_numpy(dtype=np.float32)
        label = torch.tensor(label, dtype=torch.float32)

        return spec, label


In [8]:
train_dataset = InstrumentDataset(data, spektogramy, "train")
val_dataset = InstrumentDataset(data, spektogramy, "val")
test_dataset = InstrumentDataset(data, spektogramy, "test")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

### Wybór urządzenia do obliczeń dla modelu

In [9]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    return device

### Architektura sieci

In [10]:
# jest to wstępna, jak będą dane z mel-spektogramu to się rozbuduje sieć, aby jak najlepiej zoptymalizować ją

class Instrument_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1)
        self.conv2 = nn.Conv2d(32,64,3,1)
        self.conv3 = nn.Conv2d(64,128,3,1)
        self.dropout2 = nn.Dropout2d(0.4)
        self.dropout3 = nn.Dropout2d(0.6)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc3 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout3(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc3(x)

        return x


In [11]:
device = get_device()

model = Instrument_CNN().to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Urządzenie:", device)

Urządzenie: mps


In [12]:
X, y = next(iter(train_loader))

print("Spektrogramy:", X.shape)
print("Etykiety:", y.shape)
print("Przykładowe etykiety:", y[0])

Spektrogramy: torch.Size([4, 1, 80, 346])
Etykiety: torch.Size([4, 7])
Przykładowe etykiety: tensor([0., 0., 0., 1., 0., 0., 0.])


In [13]:
epochs = 20

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )


Epoka 1/20 | Train loss: 2.5662 | Val loss: 0.6284
Epoka 2/20 | Train loss: 1.8762 | Val loss: 0.6548
Epoka 3/20 | Train loss: 1.1380 | Val loss: 0.7670
Epoka 4/20 | Train loss: 0.9121 | Val loss: 0.7482
Epoka 5/20 | Train loss: 0.7069 | Val loss: 0.7260
Epoka 6/20 | Train loss: 0.7180 | Val loss: 0.6787
Epoka 7/20 | Train loss: 0.8516 | Val loss: 0.6473
Epoka 8/20 | Train loss: 0.7143 | Val loss: 0.6296
Epoka 9/20 | Train loss: 0.6204 | Val loss: 0.6295
Epoka 10/20 | Train loss: 0.5940 | Val loss: 0.6370
Epoka 11/20 | Train loss: 0.5330 | Val loss: 0.6158
Epoka 12/20 | Train loss: 0.5819 | Val loss: 0.6023
Epoka 13/20 | Train loss: 0.6928 | Val loss: 0.6041
Epoka 14/20 | Train loss: 0.6092 | Val loss: 0.6132
Epoka 15/20 | Train loss: 0.6065 | Val loss: 0.6117
Epoka 16/20 | Train loss: 0.6115 | Val loss: 0.6145
Epoka 17/20 | Train loss: 0.5755 | Val loss: 0.6075
Epoka 18/20 | Train loss: 0.5722 | Val loss: 0.5989
Epoka 19/20 | Train loss: 0.6322 | Val loss: 0.6026
Epoka 20/20 | Train l

In [14]:
model.eval()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        prawdopodobienstwa = torch.sigmoid(logits)
        predykcje = (prawdopodobienstwa >= 0.5).float()

        print("Prawdopodobieństwa:")
        print(prawdopodobienstwa.cpu())

        print("\nPrzewidywane etykiety:")
        print(predykcje.cpu())

        print("\nPrawdziwe etykiety:")
        print(y.cpu())

        break


Prawdopodobieństwa:
tensor([[0.3836, 0.1865, 0.4849, 0.8722, 0.5968, 0.4224, 0.2045],
        [0.3797, 0.1787, 0.4848, 0.8838, 0.5875, 0.4328, 0.1995],
        [0.3854, 0.1864, 0.4836, 0.8762, 0.5857, 0.4348, 0.2062],
        [0.3784, 0.1762, 0.4836, 0.8870, 0.5863, 0.4331, 0.1963]])

Przewidywane etykiety:
tensor([[0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0.]])

Prawdziwe etykiety:
tensor([[1., 1., 1., 1., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 1., 0., 1., 0.],
        [1., 0., 0., 1., 1., 0., 0.]])
